# Payment Transaction Building

This notebook demonstrates how to build payment transactions using the Writer MCP service.

In [1]:
import sys
sys.path.append('../00_setup')
from helpers import call_api, check_health, format_algo, short_address
from config import WRITER_URL, ALICE_ADDRESS, BOB_ADDRESS, ONE_ALGO, HALF_ALGO

## Health Check

First, let's check if the Writer MCP service is running:

In [2]:
health = await check_health(WRITER_URL)
print(f"Writer MCP Service: {'✅ Healthy' if health else '❌ Not available'}")

if not health:
    print("💡 Run 'node notebooks/demo-writer-service.js' to start the real Writer service")
    print("💡 Or set USE_MOCK_MODE=True in config.py for testing")

Writer MCP Service: ✅ Healthy


## Build a Simple Payment Transaction

Let's build a payment transaction from Alice to Bob:

In [3]:
# Build payment transaction
transaction_data = {
    "fromAddress": ALICE_ADDRESS,
    "toAddress": BOB_ADDRESS,
    "microAlgos": ONE_ALGO,  # 1 ALGO
    "note": "Payment from Alice to Bob"
}

result = await call_api(f"{WRITER_URL}/tools/build_payment_transaction", transaction_data)

if result.get("success"):
    print("✅ Transaction built successfully!")
    print(f"💰 Amount: {format_algo(ONE_ALGO)} ALGO")
    print(f"👩 From: {short_address(ALICE_ADDRESS)}")
    print(f"👨 To: {short_address(BOB_ADDRESS)}")
    print(f"🆔 Transaction ID: {result.get('txId')}")
    print(f"💸 Fee: {result.get('fee', 0)} microAlgos")
    print(f"🔢 First Round: {result.get('firstRound')}")
    print(f"🔢 Last Round: {result.get('lastRound')}")
    
    if result.get('note'):
        print(f"📝 Note: {result['note']}")
        
    print(f"\n📄 Unsigned Transaction (Base64): {result.get('unsignedTxnBase64', 'N/A')[:50]}...")
else:
    print(f"❌ Error: {result.get('error')}")
    if result.get('mock_available'):
        print("💡 Set USE_MOCK_MODE=True in config.py to use mock data")

✅ Transaction built successfully!
💰 Amount: 1.0 ALGO
👩 From: 7ZUECA7H...THAIOF6Q
👨 To: GD64YIY3...CDBBHU5A
🆔 Transaction ID: CWIDERQFOIYQV2PVDJLNVG6ITHGGKBWKS5CWTENYIDI34VMBECDQ
💸 Fee: 1000 microAlgos
🔢 First Round: 55716952
🔢 Last Round: 55717952
📝 Note: This is a REAL testnet transaction. It will execute if signed and submitted.

📄 Unsigned Transaction (Base64): iqNhbXTOAA9CQKNmZWXNA+iiZnbOA1IsWKNnZW6sdGVzdG5ldC...


## Build Multiple Transactions

Let's build several different payment transactions to test various amounts:

In [4]:
# Different transaction scenarios
transactions = [
    {
        "name": "Small Payment",
        "fromAddress": ALICE_ADDRESS,
        "toAddress": BOB_ADDRESS,
        "microAlgos": HALF_ALGO,
        "note": "Small payment test"
    },
    {
        "name": "Large Payment",
        "fromAddress": BOB_ADDRESS,
        "toAddress": ALICE_ADDRESS,
        "microAlgos": ONE_ALGO * 5,  # 5 ALGO
        "note": "Large payment test"
    },
    {
        "name": "Minimal Payment",
        "fromAddress": ALICE_ADDRESS,
        "toAddress": BOB_ADDRESS,
        "microAlgos": 1000,  # 0.001 ALGO
        "note": "Minimal payment test"
    }
]

print("🔄 Building Multiple Transactions")
print("=" * 50)

for i, tx in enumerate(transactions, 1):
    print(f"\n{i}. {tx['name']}")
    print(f"   Amount: {format_algo(tx['microAlgos'])} ALGO")
    print(f"   From: {short_address(tx['fromAddress'])}")
    print(f"   To: {short_address(tx['toAddress'])}")
    
    result = await call_api(f"{WRITER_URL}/tools/build_payment_transaction", tx)
    
    if result.get("success"):
        print(f"   ✅ Success - TX ID: {result.get('txId')}")
        print(f"   💸 Fee: {result.get('fee', 0)} microAlgos")
    else:
        print(f"   ❌ Error: {result.get('error')}")

🔄 Building Multiple Transactions

1. Small Payment
   Amount: 0.5 ALGO
   From: 7ZUECA7H...THAIOF6Q
   To: GD64YIY3...CDBBHU5A
   ✅ Success - TX ID: BWHS6JCQL7FZSN74RWSRPJ3HNI6DAUR2ZS4CCYSKEHMVU6IQTYWA
   💸 Fee: 1000 microAlgos

2. Large Payment
   Amount: 5.0 ALGO
   From: GD64YIY3...CDBBHU5A
   To: 7ZUECA7H...THAIOF6Q
   ✅ Success - TX ID: KGJSYXXV7TZTGZ5QGGCGR2SSHKPPJ2HMWDNZSZT3H4N3QTTV4IHA
   💸 Fee: 1000 microAlgos

3. Minimal Payment
   Amount: 0.001 ALGO
   From: 7ZUECA7H...THAIOF6Q
   To: GD64YIY3...CDBBHU5A
   ✅ Success - TX ID: 7X4IKOQBTTI2M3B6Y6265FHBXTKFBEE3NPM2PPBGSKKWE567I2JQ
   💸 Fee: 1000 microAlgos


## Transaction with Custom Note

Let's create a transaction with a custom note field:

In [5]:
import datetime

# Create transaction with timestamp note
timestamp = datetime.datetime.now().isoformat()
custom_note = f"Payment created at {timestamp} via MCP notebook"

transaction_data = {
    "fromAddress": ALICE_ADDRESS,
    "toAddress": BOB_ADDRESS,
    "microAlgos": ONE_ALGO * 2,  # 2 ALGO
    "note": custom_note
}

print(f"📝 Creating transaction with custom note:")
print(f"   Note: {custom_note}")

result = await call_api(f"{WRITER_URL}/tools/build_payment_transaction", transaction_data)

if result.get("success"):
    print("\n✅ Transaction with custom note built successfully!")
    print(f"🆔 Transaction ID: {result.get('txId')}")
    print(f"💰 Amount: {format_algo(transaction_data['microAlgos'])} ALGO")
    print(f"📅 Timestamp: {timestamp}")
else:
    print(f"\n❌ Error: {result.get('error')}")

📝 Creating transaction with custom note:
   Note: Payment created at 2025-09-19T23:03:34.926373 via MCP notebook

✅ Transaction with custom note built successfully!
🆔 Transaction ID: ZEFQFRUQMMTAQNIWWYP7MBMARI2SAE3CI6JCYUVLXVM2A4M3SG6Q
💰 Amount: 2.0 ALGO
📅 Timestamp: 2025-09-19T23:03:34.926373


## Error Handling

Let's test error scenarios to see how the service handles invalid inputs:

In [6]:
# Test invalid address
print("🧪 Testing Error Scenarios")
print("=" * 30)

# 1. Invalid address
print("\n1. Invalid recipient address:")
result = await call_api(f"{WRITER_URL}/tools/build_payment_transaction", {
    "fromAddress": ALICE_ADDRESS,
    "toAddress": "INVALID_ADDRESS",
    "microAlgos": ONE_ALGO,
    "note": "Test invalid address"
})
print(f"   Result: {'✅ Success' if result.get('success') else '❌ Error: ' + result.get('error', 'Unknown')}")

# 2. Missing required field
print("\n2. Missing required field (toAddress):")
result = await call_api(f"{WRITER_URL}/tools/build_payment_transaction", {
    "fromAddress": ALICE_ADDRESS,
    "microAlgos": ONE_ALGO,
    "note": "Test missing field"
})
print(f"   Result: {'✅ Success' if result.get('success') else '❌ Error: ' + result.get('error', 'Unknown')}")

# 3. Zero amount
print("\n3. Zero amount:")
result = await call_api(f"{WRITER_URL}/tools/build_payment_transaction", {
    "fromAddress": ALICE_ADDRESS,
    "toAddress": BOB_ADDRESS,
    "microAlgos": 0,
    "note": "Test zero amount"
})
print(f"   Result: {'✅ Success' if result.get('success') else '❌ Error: ' + result.get('error', 'Unknown')}")

🧪 Testing Error Scenarios

1. Invalid recipient address:
   Result: ❌ Error: address seems to be malformed

2. Missing required field (toAddress):
   Result: ❌ Error: Missing required parameters: fromAddress, toAddress, microAlgos

3. Zero amount:
   Result: ❌ Error: Missing required parameters: fromAddress, toAddress, microAlgos
